# Pruebas Estadísticas

### 1. Carga de librerias a utilizar

In [10]:
!pip install pandas numpy scipy scikit-posthocs


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 654.1 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 769.4 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 469.0 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 612.1 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.0/474.0 kB 522.7 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 301.8 kB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### 2. Cargamos las librerías a utilizar

In [4]:
import numpy as np
import pandas as pd
from scipy import stats

### 3. Carga de datos

In [5]:
df_biare = pd.read_csv("../Limpieza de datos (Practica 1)/biare_limpio_2021_2024.csv")

df_biare['N_ENT'] = df_biare['N_ENT'].astype(str)
df_biare['N_REN'] = df_biare['N_REN'].astype(str)

### 4. Supuesto de normalidad

Utilizamos el supuesto de normalidad para decidir si usamos ANOVA O Kruskal-wallis, usando la prueba de Shapiro-Wilk, sabemos si los resultados de la pregunta 1 de BIARE tiene una distribución normal.

Si el p-valor de la prueba es mayor a 0.05, entonces tiene una distribución normal, si es menor a 0.05 entonces no tiene una distribución normal.

El resultado estadístico entre más cercano este a 1, más normal es la distribución.

In [6]:
# Shapiro-Wilk sobre una muestra de 500 personas, de la pregunta cb_P1
muestra = df_biare['cb_P1'].dropna().sample(500, random_state=42) #quitamos los nulos si hay y tomamos una muestra de 500 personas permitiendo repeticion con random_state
stat, p_valor = stats.shapiro(muestra)

print(f"Estadístico: {stat:.4f}, p-valor: {p_valor:.10f}")

Estadístico: 0.8088, p-valor: 0.0000000000


Como el p-valor es practicamente 0, eso quiere decir que p-valor < 0.05, por lo que no es una distribución normal, utilizaremos Kruskal-Wallis

### 5. Realizamos la prueba de Kruskal-Wallis

Realizamos la prueba de Kruskal-Wallis, para ver si hay alguna diferencía entre las respuestas de la satisfacción con la vida entre los años

Si el p-valor es menor a 0.05 entonces sí hay una diferencia significativa entre al menos un año, si el p-valor es mayor a 0.05, entonces no hay una diferencia significativa entre los años.

In [8]:
#cargamos los anios de cuestionarios que aarecen en la muestra
a2021 = df_biare[df_biare['ANIO'] == 2021]['cb_P1'].dropna()
a2022 = df_biare[df_biare['ANIO'] == 2022]['cb_P1'].dropna()
a2023 = df_biare[df_biare['ANIO'] == 2023]['cb_P1'].dropna()
a2024 = df_biare[df_biare['ANIO'] == 2024]['cb_P1'].dropna()

#calculamos el estadistico y el p-valor de la prueba de Kruskal-Wallis
stat, p_valor = stats.kruskal(a2021, a2022, a2023, a2024)

#mostramos el estadistico y el p-valor de la prueba de Kruskal-Wallis
print(f"Estadístico H: {stat:.4f}")
print(f"p-valor: {p_valor:.10f}")

Estadístico H: 119.5684
p-valor: 0.0000000000


Como el p-valor es 0, entonces el p-valor < 0.05, por lo tanto si hay una diferencia entre al menos un año con respecto a la satisfacción con la vida

##### Prueba de Dunn

Realizamos una prueba de Dunn, para saber entre cuales años hay exactamente una diferencia significante

In [11]:
import scikit_posthocs as sp

resultado_dunn = sp.posthoc_dunn(df_biare, val_col='cb_P1', group_col='ANIO', p_adjust='bonferroni')
print(resultado_dunn)

              2021          2022          2023          2024
2021  1.000000e+00  1.000000e+00  1.000000e+00  2.703482e-16
2022  1.000000e+00  1.000000e+00  1.000000e+00  2.591205e-20
2023  1.000000e+00  1.000000e+00  1.000000e+00  4.907294e-18
2024  2.703482e-16  2.591205e-20  4.907294e-18  1.000000e+00


Podemos observar que 2024 es el que difiere de los demas años, siendo el único que al compararlo con los demás años, da un resultado cercano a 0 en su p-valor, por lo que nuestro hallazgo es que 2024, difiere de los demás años.